In [1]:
import torch
import requests
from PIL import Image
from io import BytesIO
from transformers import AutoModel, AutoTokenizer


from utils.model_utils import load_hf_model_and_processor_or_tokenizer

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
model_name = "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct"
print(f"Loading model: {model_name}...")

# Use the utility function to load the model and processor
# It handles InternVL specific loading requirements
model, processor = load_hf_model_and_processor_or_tokenizer(
    model_name_or_path=model_name,
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

# InternVL processor is essentially the tokenizer, or contains it
tokenizer = processor
if hasattr(processor, 'tokenizer'):
    tokenizer = processor.tokenizer

# Set img_context_token_id on the model as required by get_combined_embeddings
print(f"Model loaded successfully.")

Loading model: lmms-lab/LLaVA-OneVision-1.5-8B-Instruct...

Loading lmms-lab/LLaVA-OneVision-1.5-8B-Instruct processor...

Loading lmms-lab/LLaVA-OneVision-1.5-8B-Instruct model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully.


In [ ]:
from typing import List, Optional, Tuple, Union
from dataclasses import dataclass
from transformers.modeling_outputs import BaseModelOutputWithPast, ModelOutput
from transformers.utils import is_torchdynamo_compiling
from transformers.cache_utils import DynamicCache, logger

@dataclass
class LLaVAOneVision1_5_ModelOutputWithPast(ModelOutput):
    """
    Base class for Llava outputs, with hidden states and attentions.
    Args:
        last_hidden_state (`torch.FloatTensor` of shape `(batch_size, sequence_length, hidden_size)`):
            Sequence of hidden-states at the output of the last layer of the model.
        past_key_values (`tuple(tuple(torch.FloatTensor))`, *optional*, returned when `use_cache=True` is passed or when `config.use_cache=True`):
            Tuple of `tuple(torch.FloatTensor)` of length `config.n_layers`, with each tuple having 2 tensors of shape
            `(batch_size, num_heads, sequence_length, embed_size_per_head)`)
            Contains pre-computed hidden-states (key and values in the self-attention blocks) that can be used (see
            `past_key_values` input) to speed up sequential decoding.
        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.
            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.
        attentions (`tuple(torch.FloatTensor)`, *optional*, returned when `output_attentions=True` is passed or when `config.output_attentions=True`):
            Tuple of `torch.FloatTensor` (one for each layer) of shape `(batch_size, num_heads, sequence_length,
            sequence_length)`.
            Attentions weights after the attention softmax, used to compute the weighted average in the self-attention
            heads.
        rope_deltas (`torch.LongTensor` of shape `(batch_size, )`, *optional*):
            The rope index difference between sequence length and multimodal rope.
    """

    last_hidden_state: torch.FloatTensor = None
    past_key_values: Optional[List[torch.FloatTensor]] = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None
    rope_deltas: Optional[torch.LongTensor] = None

def llava_forward(
        self,
        input_ids: torch.LongTensor = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        pixel_values: Optional[torch.Tensor] = None,
        pixel_values_videos: Optional[torch.FloatTensor] = None,
        image_grid_thw: Optional[torch.LongTensor] = None,
        video_grid_thw: Optional[torch.LongTensor] = None,
        rope_deltas: Optional[torch.LongTensor] = None,
        cache_position: Optional[torch.LongTensor] = None,
    ) -> Union[Tuple, LLaVAOneVision1_5_ModelOutputWithPast]:
        r"""
        pixel_values_videos (`torch.FloatTensor` of shape `(seq_length, num_channels * temporal_size * image_size * image_size)):
            The tensors corresponding to the input videos. Pixel values can be obtained using
            [`AutoImageProcessor`]. See [`Qwen2VLImageProcessor.__call__`] for details. [`Qwen2VLProcessor`] uses
            [`Qwen2VLImageProcessor`] for processing videos.
        image_grid_thw (`torch.LongTensor` of shape `(num_images, 3)`, *optional*):
            The temporal, height and width of feature shape of each image in LLM.
        video_grid_thw (`torch.LongTensor` of shape `(num_videos, 3)`, *optional*):
            The temporal, height and width of feature shape of each video in LLM.
        rope_deltas (`torch.LongTensor` of shape `(batch_size, )`, *optional*):
            The rope index difference between sequence length and multimodal rope.
        """
        print("In forward")
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if inputs_embeds is None:
            inputs_embeds = self.get_input_embeddings()(input_ids)
            if pixel_values is not None:
                image_embeds = self.get_image_features(pixel_values, image_grid_thw)
                n_image_tokens = (input_ids == self.config.image_token_id).sum().item()
                n_image_features = image_embeds.shape[0]
                if not is_torchdynamo_compiling() and n_image_tokens != n_image_features:
                    raise ValueError(
                        f"Image features and image tokens do not match: tokens: {n_image_tokens}, features {n_image_features}"
                    )
                image_mask = (
                    (input_ids == self.config.image_token_id)
                    .unsqueeze(-1)
                    .expand_as(inputs_embeds)
                    .to(inputs_embeds.device)
                )
                image_embeds = image_embeds.to(inputs_embeds.device, inputs_embeds.dtype)
                inputs_embeds = inputs_embeds.masked_scatter(image_mask, image_embeds)

            if pixel_values_videos is not None:
                video_embeds = self.get_video_features(pixel_values_videos, video_grid_thw)
                n_video_tokens = (input_ids == self.config.video_token_id).sum().item()
                n_video_features = video_embeds.shape[0]
                if not is_torchdynamo_compiling() and n_video_tokens != n_video_features:
                    raise ValueError(
                        f"Video features and video tokens do not match: tokens: {n_video_tokens}, features {n_video_features}"
                    )
                video_mask = (
                    (input_ids == self.config.video_token_id)
                    .unsqueeze(-1)
                    .expand_as(inputs_embeds)
                    .to(inputs_embeds.device)
                )
                video_embeds = video_embeds.to(inputs_embeds.device, inputs_embeds.dtype)
                inputs_embeds = inputs_embeds.masked_scatter(video_mask, video_embeds)

            if attention_mask is not None:
                attention_mask = attention_mask.to(inputs_embeds.device)
        if (input_ids == self.config.image_token_id).any():
            image_limits = []
            for i in range(input_ids.shape[0]):
                indices = torch.where(input_ids[i] == self.config.image_token_id)[0]
                if indices.numel() > 0:
                    image_limits.append((indices[0].item(), indices[-1].item()))
                else:
                    image_limits.append(None)
                self.image_attention_mask = (input_ids != self.config.image_token_id).to(
                    torch.int
                )
        else:
            image_limits = None
            if self.image_attention_mask is not None:
                self.image_attention_mask = torch.cat(
                    (
                        self.image_attention_mask,
                        torch.ones(self.image_attention_mask.shape[:-1] + (1,)).to(
                            self.image_attention_mask.device
                        ),
                    ),
                    dim=-1,
                )
            else:
                self.image_attention_mask = None
        if use_cache and past_key_values is None:
            past_key_values = DynamicCache()

        if cache_position is None:
            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
            cache_position = torch.arange(
                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device
            )

        if position_ids is None:
            position_ids = cache_position.unsqueeze(0)


        outputs = self.language_model(
            input_ids=None,
            position_ids=position_ids,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
            cache_position=cache_position,
            image_limits=image_limits,
            image_attention_mask=self.image_attention_mask,
        )

        output = LLaVAOneVision1_5_ModelOutputWithPast(
            last_hidden_state=outputs.last_hidden_state,
            past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
            rope_deltas=self.rope_deltas,
        )
        return output if return_dict else output.to_tuple()

def llama_forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[List[torch.FloatTensor]] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> Union[Tuple, BaseModelOutputWithPast]:
        image_limits = kwargs.get("image_limits", None)
        image_attention_mask = kwargs.get("image_attention_mask", None)
        layers_to_skip = kwargs.get("layers_to_skip", [])
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        use_cache = use_cache if use_cache is not None else self.config.use_cache

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if (input_ids is None) ^ (inputs_embeds is not None):
            raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

        if self.gradient_checkpointing and self.training:
            if use_cache:
                logger.warning_once(
                    "`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`..."
                )
                use_cache = False

        # torch.jit.trace() doesn't support cache objects in the output
        if use_cache and past_key_values is None and not torch.jit.is_tracing():
            past_key_values = DynamicCache()

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)

        if cache_position is None:
            past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
            cache_position = torch.arange(
                past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device
            )

        # the hard coded `3` is for temporal, height and width.
        if position_ids is None:
            position_ids = cache_position.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
        # elif position_ids.dim() == 2: # 这是为了3drope准备的
        #     position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

        causal_mask = self._update_causal_mask(
            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions
        )
        image_mask = self._update_causal_mask(
            image_attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions
        )
        if causal_mask is not None and image_mask is not None:
            image_mask = torch.minimum(image_mask, causal_mask)
        hidden_states = inputs_embeds

        # create position embeddings to be shared across the decoder layers
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        # decoder layers
        all_hidden_states = () if output_hidden_states else None
        all_self_attns = () if output_attentions else None
        next_decoder_cache = None
        if image_limits is not None:
            self.image_limits = image_limits
        min_dtype = torch.finfo(image_mask.dtype).min if torch.is_floating_point(image_mask) else False
        for i, decoder_layer in enumerate(self.layers):
            layer_mask = causal_mask
            if output_hidden_states:
                all_hidden_states += (hidden_states,)

            if i in layers_to_skip and self.image_limits is not None:
                print("skipping layer", i)
                if i == layers_to_skip[0]:
                    stored_embeds = hidden_states
                layer_mask = image_mask.clone()
                for b, limit in enumerate(self.image_limits):
                    if limit is not None:
                        start, end = limit
                        if layer_mask.shape[2] > 1:
                            layer_mask[b, :, end + 1 :, start : end + 1] = min_dtype
                        else:
                            layer_mask[b, :, :, start : end + 1] = min_dtype

            elif (
                len(layers_to_skip) > 0
                and i == (layers_to_skip[-1] + 1)
                and self.image_limits is not None
            ):
                for b, limit in enumerate(self.image_limits):
                    if limit is not None:
                        start, end = limit
                        hidden_states[b, start : end + 1, :] = stored_embeds[
                            b, start : end + 1, :
                        ]
            if self.gradient_checkpointing and self.training:
                layer_outputs = self._gradient_checkpointing_func(
                    decoder_layer.__call__,
                    hidden_states,
                    layer_mask,
                    position_ids,
                    past_key_values,
                    output_attentions,
                    use_cache,
                    cache_position,
                    position_embeddings,
                )
            else:
                layer_outputs = decoder_layer(
                    hidden_states,
                    attention_mask=layer_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_values,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                    cache_position=cache_position,
                    position_embeddings=position_embeddings,
                )

            hidden_states = layer_outputs[0]

            if use_cache:
                next_decoder_cache = layer_outputs[2 if output_attentions else 1]

            if output_attentions:
                all_self_attns += (layer_outputs[1],)

        hidden_states = self.norm(hidden_states)

        # add hidden states from the last decoder layer
        if output_hidden_states:
            all_hidden_states += (hidden_states,)

        next_cache = next_decoder_cache if use_cache else None

        if not return_dict:
            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)
        return BaseModelOutputWithPast(
            last_hidden_state=hidden_states,
            past_key_values=next_cache,
            hidden_states=all_hidden_states,
            attentions=all_self_attns,
        )


In [ ]:
from functools import partialmethod

model.model.__class__.forward = llava_forward
model.language_model.__class__.forward = partialmethod(llama_forward, layers_to_skip=range(22))


In [8]:
def load_image_from_url(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return None

# Tuple of (Image URL, Question)
data_samples = [
    (
        "http://images.cocodataset.org/val2017/000000039769.jpg",
        "Describe the scene in detail."
    ),
    (
        "http://images.cocodataset.org/val2017/000000000139.jpg",
        "What appliances can you see in this room?"
    )
]

images = []
questions = []

for url, question in data_samples:
    img = load_image_from_url(url)
    if img:
        images.append(img)
        questions.append(question)
        print(f"Loaded image for question: '{question}'")

    

Loaded image for question: 'Describe the scene in detail.'
Loaded image for question: 'What appliances can you see in this room?'


In [9]:
torch.set_printoptions(threshold=10_000)

samples = [
    (
        "http://images.cocodataset.org/val2017/000000039769.jpg",
        "Describe the scene in detail.",
    ),
    # (
    #     "http://images.cocodataset.org/val2017/000000000139.jpg",
    #     "What appliances can you see in this room?",
    # ),
]

images = []
questions = []
for i, (url, question) in enumerate(samples, start=1):
    image = load_image_from_url(url)
    if image is None:
        print(f"[{i}] Failed to load image: {url}")
        continue
    images.append(image)
    questions.append(question)

if len(images) > 2:
    print(f"Expected batch size 2, but got {len(images)} loaded images.")
else:
    conversations = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": question},
                ],
            }
        ]
        for question in questions
    ]

    prompts = [
        processor.apply_chat_template(conversation, add_generation_prompt=True)
        for conversation in conversations
    ]

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(device)

    generate_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )
    input_lengths = inputs["attention_mask"].sum(dim=1).tolist()

    for i, question in enumerate(questions, start=1):
        answer_ids = generate_ids[i - 1, input_lengths[i - 1] :]
        answer = processor.decode(answer_ids, skip_special_tokens=True)
        print(f"\n[{i}] Q: {question}")
        print(f"[{i}] A: {answer}")

In forward
Image mask torch.Size([1, 1, 418, 418]) tensor([[-0.0000e+00, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38,
         -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38, -3.3895e+38],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+0